# Convolutional Neural Networks: 2D and 1D

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_11/CNN_2D_1D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_11/CNN_2D_1D.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 

This notebook follows our MLP session. We will train a **2D CNN for MNIST digit classification** and a **1D CNN for Iowa house-price regression**. Each example extracts and visualizes intermediate convolutional features.

**Learning goals:** 
- Recognize kernels and feature maps
- Understand input/output shapes
- Train convolutional networks
- Compare convolutional features across image and tabular inputs.

### Documentation

- [```PyTorch```](https://pytorch.org/) 
- [```PyTorch Documentation```](https://docs.pytorch.org/docs/2.14/index.html)
- [```Keras```](https://keras.io/)

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution



## Environment

### Install dependencies

Run the following in Colab if necessary, then restart the runtime if packages were already imported with a different backend.

```python
%pip install -q -U "keras>=3" torch scikit-learn pandas matplotlib
```

The backend must be set **before** importing `keras`.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score

import keras


keras.utils.set_random_seed(42)
print("Keras:", keras.__version__, "| Backend:", keras.backend.backend())

## Understanding convolution

### From input to feature maps

A **convolutional kernel** moves across local neighborhoods, learning filters that respond to particular patterns.

- **Conv2D:** a kernel moves across image height and width. An MNIST input has shape `(28, 28, 1)`.
- **Conv1D:** a kernel moves across a sequence axis. Here we arrange selected numerical housing attributes as `(features, 1)`.
- **Channels:** a Conv2D layer may produce many feature maps even if its input has one channel; each output filter produces **one channel**. Conv1D behaves similarly.
- **Pooling:** reduces the spatial/sequence length and helps summarize local responses.

**Visualization convention:** an individual convolutional feature map is displayed as a **single channel**. For the 2D example, we also create a three-channel RGB composite from three distinct filters. A three-channel composite is a visualization, **not** a literal RGB photograph of the input.

### Key equations

For one-dimensional input with a single channel, a filter computes a weighted local sum:

$$z_{t,k}=b_k+\sum_{j=0}^{K-1}w_{j,k}x_{t+j},\qquad a_{t,k}=g(z_{t,k}).$$

A 2D filter extends this operation over **two** spatial indices. During training, backpropagation learns the kernel weights to minimize the task-specific loss.

## CNN in Two Dimensions - MNIST Classification

### Load and explore MNIST

MNIST has 28×28 grayscale handwritten-digit images with labels 0–9. We normalize pixel intensities to `[0, 1]` and add an explicit **single channel** dimension.

In [ ]:
data_train = _
data_test  = _

X_train, y_train = _ , _ 
X_test, y_test = _ , _ 

X_train = (X_train.values.reshape(-1, 28,28,1).astype("float32") / 255.0)
X_test = (X_test.values.reshape(-1, 28,28,1).astype("float32") / 255.0)
print("Train:", X_train.shape, "| Test:", X_test.shape)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.flat, range(10)):
    ax.imshow(X_train[i, :, :, 0], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Build a 2D CNN

The hidden convolutional layers learn image features. The final **softmax** layer outputs probabilities for 10 classes. We use **sparse categorical cross-entropy** because the labels are integer class IDs.

In [ ]:
cnn2d = keras.Sequential([
    keras.Input(shape=( _ , _ , 1)),
    keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu", name="conv2d_first"),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu", name="conv2d_second"),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

cnn2d.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), 
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cnn2d.summary()

### Train and evaluate

To keep this practical for Colab, we use a **subset of the MNIST training set**. The test set remains unchanged.

In [ ]:
rng = np.random.default_rng(42)

train_subset = rng.choice(len(X_train), size=5000, replace=False)
history2d = cnn2d.fit( _ , _ ,
                      validation_split=0.10, epochs=10, 
                      batch_size=128, verbose=1)

In [ ]:
test_loss2d, test_accuracy2d = cnn2d.evaluate(X_test, y_test, verbose=0)

print(f"Test accuracy: {test_accuracy2d:.3f} | Test cross-entropy: {test_loss2d:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key in ["loss", "val_loss"]: axes[0].plot(history2d.history[key], label=key)
for key in ["accuracy", "val_accuracy"]: axes[1].plot(history2d.history[key], label=key)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy")
for ax in axes: ax.legend()
plt.tight_layout()
plt.show()

### Visualize predictions

Compare a few handwritten digits with the model's predictions.

In [ ]:
sample_ids = np.arange(10)
probabilities2d = cnn2d.predict(X_test[sample_ids], verbose=0)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i, predicted in zip(axes.flat, sample_ids, probabilities2d.argmax(axis=1)):
    ax.imshow(X_test[i, :, :, 0], cmap="gray")
    ax.set_title(f"True {y_test[i]} · Pred {predicted}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Extract 2D convolutional features

An intermediate Keras model returns the **feature maps** from the first convolutional layer. Its output has shape `(batch, height, width, filters)`. We show six **one-channel** maps separately, then create an RGB composite from three maps.

In [ ]:
feature_extractor2d = keras.Model(inputs=cnn2d.inputs, outputs=cnn2d.get_layer("conv2d_first").output)

image = X_test[0:1]
feature_maps2d = feature_extractor2d.predict(image, verbose=0)[0]
print("Input image:", image.shape, "| Convolutional maps:", feature_maps2d.shape)


In [ ]:

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes[0, 0].imshow(image[0, :, :, 0], cmap="gray")
axes[0, 0].set_title("Original: 1 channel")
for ax, channel in zip(axes.flat[1:7], range(6)):
    ax.imshow(feature_maps2d[:, :, channel], cmap="viridis")
    ax.set_title(f"Filter {channel}: 1 channel")
for ax in axes.flat: ax.axis("off")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

rgb_maps = feature_maps2d[:, :, :3]
rgb_maps = (rgb_maps - rgb_maps.min(axis=(0, 1), keepdims=True)) / (np.ptp(rgb_maps, axis=(0, 1), keepdims=True) + 1e-8)
plt.figure(figsize=(4, 4))
plt.imshow(rgb_maps)
plt.title("Three-filter RGB composite (3 channels)")
plt.axis("off")
plt.show()

### Explore the meaning of filters

Different filters respond to different patterns. Inspect how the maps change when you select another digit or switch from `conv2d_first` to `conv2d_second`.

**Question:** Are the filters identifying recognizable digit parts, or more general edges and textures?

## CNN in One Dimension - California Housing

### Why Conv1D on housing data?

Conv1D normally excels when **nearby positions have meaningful relationships**, as in sensor signals or time series. Iowa housing records are tabular: the ordering of columns is chosen by us, so convolution is **not automatically an appropriate inductive bias**. We use it to demonstrate Conv1D tensor shapes and feature maps, then encourage comparison with the MLP from the previous session.

In [ ]:
houses = _ 

print("Dataset: California Housing | Shape:", houses.shape)
print("Columns containing price:", [col for col in houses.columns if "value" in col.lower()])
display(houses.head(3))

### Select features and explore the target

We use a small numerical feature subset to keep the architecture understandable. The target is **SalePrice**. A log transformation reduces price skew and makes optimization more stable.

In [ ]:
house_features = ["longitude", "latitude", "housing_median_age", "total_rooms",
                  "total_bedrooms", "population", "households", "median_income"]

X_house = houses[house_features].apply(pd.to_numeric, errors="coerce")
y_house = pd.to_numeric(houses["median_house_value"], errors="coerce")

valid = y_house.notna() & (y_house > 0) 
X_house, y_house = X_house.loc[valid], y_house.loc[valid].to_numpy(dtype="float64")

print("Predictor order:", house_features)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(y_house, bins=35)
axes[0].set(xlabel="Median house value ($)", ylabel="Count", title="House values")
axes[1].hist(np.log1p(y_house), bins=35)
axes[1].set(xlabel="log(1 + Median house value)", ylabel="Count", title="Log house values")
plt.tight_layout()
plt.show()

### Prepare tensors for Conv1D

We split before imputing or scaling, so **test data does not influence preprocessing**. The input shape is `(samples, sequence_length, channels)` = `(n, 8, 1)`.

For regression, the final layer will have **one linear output** and use **MSE**.

In [ ]:
Xh_train, Xh_test, yh_train, yh_test = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

scaler = StandardScaler()
Xh_train = scaler.fit_transform(Xh_train).astype("float32")[..., np.newaxis]
Xh_test = scaler.transform(Xh_test).astype("float32")[..., np.newaxis]

yh_train = yh_train.astype("float32")
yh_test = yh_test.astype("float32")

print("Conv1D train input:", Xh_train.shape, "| Target:", yh_train.shape)

### Build a 1D CNN

`Conv1D` learns filters spanning adjacent entries in the chosen feature order. `GlobalAveragePooling1D` summarizes the resulting feature maps. The final Dense neuron predicts a **standardized log-price**.

In [ ]:
cnn1d = keras.Sequential([
    keras.Input(shape=(len(house_features), 1)),
    keras.layers.Conv1D(16, kernel_size=3, padding="same", activation="relu", name="conv1d_first"),
    keras.layers.Conv1D(32, kernel_size=3, padding="same", activation="relu", name="conv1d_second"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1)
])

cnn1d.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), 
              loss="mse", metrics=["mae"])
cnn1d.summary()

### Train and evaluate the regression CNN

Early stopping monitors validation MSE. To interpret performance, invert the log and standardization transformations and compute metrics in **dollars**.

In [ ]:
history1d = cnn1d.fit( _ , _ ,
                      validation_split=0.2, epochs=200, 
                      batch_size=1000, verbose=1)


In [ ]:

predicted_prices = cnn1d.predict(Xh_test, verbose=0)

rmse = np.sqrt(mean_squared_error(yh_test, predicted_prices))
mae = mean_absolute_error(yh_test, predicted_prices)
r2 = r2_score(yh_test, predicted_prices)

print(f"Test RMSE: ${rmse:,.0f} | MAE: ${mae:,.0f} | R²: {r2:.3f}")

# Training and validation loss
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history1d.history["loss"], label="Training loss")
axes[0].plot(history1d.history["val_loss"], label="Validation loss")
axes[0].set(xlabel="Epoch", ylabel="MSE", title="Training history")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Actual vs predicted values
axes[1].scatter(yh_test, predicted_prices, alpha=0.4, s=15)
limits = [min(yh_test.min(), predicted_prices.min()), max(yh_test.max(), predicted_prices.max())]
axes[1].plot(limits, limits, "r--", label="Perfect prediction")
axes[1].set(xlabel="Actual house value ($)", ylabel="Predicted house value ($)",
            title=f"Actual vs Predicted\nRMSE: ${rmse:,.0f} | R²: {r2:.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

### Extract 1D convolutional features

The convolutional output has shape `(batch, positions, filters)`. Each filter produces **one 1D feature map** across the eight input positions. Plot each map separately to preserve the requested **single-channel visualization**.

> **Interpretation:** The horizontal axis refers to the **chosen ordering of input columns**, not time or physical distance. Convolutional features may reflect this artificial adjacency.

In [ ]:
feature_extractor1d = keras.Model(inputs=cnn1d.inputs, outputs=cnn1d.get_layer("conv1d_first").output)

sample_house = Xh_test[:1]
feature_maps1d = feature_extractor1d.predict(sample_house, verbose=0)[0]
print("Input:", sample_house.shape, "| Feature maps:", feature_maps1d.shape)
fig, axes = plt.subplots(3, 2, figsize=(11, 8))
for channel, ax in enumerate(axes.flat):
    ax.plot(range(len(house_features)), feature_maps1d[:, channel], marker="o")
    ax.set_title(f"Filter {channel}: one-channel feature map")
    ax.set_xticks(range(len(house_features)), house_features, rotation=50, ha="right")
    ax.set_ylabel("Activation")

plt.tight_layout()
plt.show()

### Compare the raw input and a learned feature map

The original standardized attributes form one input channel. A convolutional filter transforms this into a new one-channel sequence of activations.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(sample_house[0, :, 0], marker="o")
axes[0].set_title("Input: standardized housing features (1 channel)")
axes[1].plot(feature_maps1d[:, 0], marker="o")
axes[1].set_title("Output: first convolutional filter (1 channel)")
for ax in axes:
    ax.set_xticks(range(len(house_features)), house_features, rotation=35, ha="right")
    ax.set_ylabel("Value")
plt.tight_layout()
plt.show() 

## Comparing 2D and 1D Convolutions

### Architecture and interpretation

| Property | MNIST CNN | Iowa housing CNN |
|---|---|---|
| Input shape | `(28, 28, 1)` | `(8, 1)` |
| Convolution | `Conv2D` | `Conv1D` |
| Local neighborhood | Adjacent pixels | Adjacent selected columns |
| Target | Digit class | House price |
| Output | 10-class softmax | One linear value |
| Loss | Cross-entropy | MSE |
| Feature visualization | Individual 2D maps; 3-filter RGB composite | Individual 1D curves |
| Main limitation | Relatively simple image task | Feature order has no natural locality |

### Questions and experiments

- What do **kernel size**, **number of filters**, and **pooling** change?
- Compare feature maps from the first and second MNIST convolutional layers. Which are easier to interpret?
- Change the MNIST input image and inspect how the same filters respond.
- Change the Iowa feature order **consistently for train and test** and retrain. Does performance change?
- Compare the Iowa Conv1D with the **MLP from the previous notebook** using the same split and preprocessing.
- Why does a 2D image have meaningful spatial neighborhoods while a housing table generally does not?
- How would you modify the Conv1D example for a genuine time-series or sensor signal?

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Neural Networks - Convolutional Neural Networks (1D and 2D): Machine Learning Notebook*. Institución Universitaria Pascual Bravo.